# Minimal RAG System

A minimal Retrieval-Augmented Generation (RAG) pipeline that:
1. Downloads a WHO PDF
2. Splits it into chunks
3. Embeds chunks with a sentence-transformer model
4. Stores vectors in ChromaDB
5. Retrieves relevant chunks at query time
6. Generates an answer with `flan-t5-base`

```
PDF  →  Chunks  →  Embeddings  →  VectorDB
                                      ↑
Query  →  Retrieve  →  Prompt  →  LLM  →  Answer
```

## Setup — install dependencies

In [1]:
# Uncomment and run once if packages are not yet installed
!pip install langchain langchain-community langchain-chroma langchain-huggingface \
             chromadb sentence-transformers transformers pypdf


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Imports

In [2]:
import os
os.environ["USE_TF"] = "0"
import urllib.request
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

/Users/krishnapriya/.pyenv/versions/3.10.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 0 — Download WHO PDF

In [ ]:
WHO_URL = (
    "https://iris.who.int/server/api/core/bitstreams/"
    "6989e26c-c181-4ec8-bb99-104415a2e142/content"
)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "who_document.pdf"

if not PDF_PATH.exists():
    # WHO server requires a browser-like User-Agent
    req = urllib.request.Request(
        WHO_URL,
        headers={"User-Agent": "Mozilla/5.0 (compatible; RAG-demo/1.0)"}
    )
    with urllib.request.urlopen(req) as response, open(PDF_PATH, "wb") as f:
        f.write(response.read())
    print(f"✅ Downloaded: {PDF_PATH} ({PDF_PATH.stat().st_size / 1024:.1f} KB)")
else:
    print(f"✅ Already exists: {PDF_PATH} — skipping download")

## Step 1 — Load PDF pages

In [4]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"✅ Loaded {len(pages)} pages")
print(f"\nFirst page preview:\n{pages[0].page_content[:400]}")

✅ Loaded 106 pages

First page preview:
Policies and interventions  
to create healthy school food 
environments
WHO guideline


## Step 2 — Chunk documents

Each chunk is ~500 characters with a 50-character overlap so context is not lost at boundaries.

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(pages)
print(f"✅ Created {len(chunks)} chunks from {len(pages)} pages")
print(f"\nSample chunk:\n{chunks[0].page_content}")

✅ Created 717 chunks from 106 pages

Sample chunk:
Policies and interventions  
to create healthy school food 
environments
WHO guideline


## Steps 3 & 4 — Embed and store in ChromaDB

`all-MiniLM-L6-v2` is a fast, lightweight sentence-transformer (384-dim embeddings).  
Change `device` to `"cpu"` if you don't have a GPU.

In [6]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": DEVICE}
)

VECTORSTORE_DIR = "./who_vectorstore"
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=VECTORSTORE_DIR
)
print(f"✅ Stored {vectorstore._collection.count()} vectors in ChromaDB at '{VECTORSTORE_DIR}'")

Using device: cpu
✅ Stored 717 vectors in ChromaDB at './who_vectorstore'


## Step 5 — Set up retriever

`k=3` means the retriever returns the 3 most semantically similar chunks for each query.

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ Retriever ready (top-3 chunks per query)")

✅ Retriever ready (top-3 chunks per query)


## Step 6 — Load LLM (flan-t5-base)

`flan-t5-base` is a small encoder-decoder model that runs on CPU if needed.

In [8]:
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device=0 if DEVICE == "cuda" else -1   # -1 → CPU
)
print(f"✅ Loaded LLM: {MODEL_NAME} on {DEVICE}")

Device set to use cpu


✅ Loaded LLM: google/flan-t5-base on cpu


## Helper functions

In [9]:
def format_docs(docs):
    """Format retrieved documents into a single context string."""
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page", "?")
        formatted.append(f"[Source: Page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)


def ask_rag(question, show_debug=True):
    """Run a full RAG query: retrieve → prompt → generate."""
    # Retrieve
    retrieved_docs = retriever.invoke(question)
    context = format_docs(retrieved_docs)

    # Build prompt
    prompt = f"""Answer the question based ONLY on the following context.
If the context doesn't contain enough information, say "I don't have enough information to answer this."
Do not make up information that isn't in the context.

Context:
{context}

Question: {question}

Answer:"""

    # Generate
    result = generator(prompt)[0]["generated_text"]

    if show_debug:
        print(f"\n🔎 Question: {question}")
        print(f"\n📎 Retrieved {len(retrieved_docs)} chunks:")
        for i, doc in enumerate(retrieved_docs):
            page = doc.metadata.get("page", "?")
            print(f"\n  --- Chunk {i+1} (Page {page}) ---")
            print(f"  {doc.page_content[:200]}...")
        print(f"\n{'='*60}")
        print(f"💬 Answer: {result}")
        print(f"{'='*60}")

    return result, retrieved_docs

print("✅ Helper functions defined")

✅ Helper functions defined


## Ask questions about the WHO document

In [10]:
answer, docs = ask_rag("What is the main topic of this document?")


🔎 Question: What is the main topic of this document?

📎 Retrieved 3 chunks:

  --- Chunk 1 (Page 32) ---
  3. Summary of evidence...

  --- Chunk 2 (Page 27) ---
  equity and human rights, acceptability and feasibility. The contextual factors in the review included those 
outlined in the WHO handbook for guideline development  (Chapters 10 and 18)  (67). Extra q...

  --- Chunk 3 (Page 5) ---
  Annex 5.  External contributors 54
Annex 6.  Guidance questions for the review of contextual factors 56
Annex 7.  Details of rapid review update 57
Annex 8.  GRADE evidence profiles 63
  GRADE evidenc...

💬 Answer: I don't have enough information to answer this.


In [11]:
answer, docs = ask_rag("What are the key recommendations mentioned?")


🔎 Question: What are the key recommendations mentioned?

📎 Retrieved 3 chunks:

  --- Chunk 1 (Page 48) ---
  (73) provides a foundational whole-of-school approach for implementing the guideline’s recommendations,...

  --- Chunk 2 (Page 48) ---
  31
5. Implementation considerations
This chapter is not intended to provide an exhaustive list of implementation considerations. Instead, it 
aims to highlight some key considerations for implementing...

  --- Chunk 3 (Page 16) ---
  guidance and recommendations, including from partner organizations, to ensure a coordinated approach 
to promoting healthy eating habits, improving nutrition and enhancing the well-being of children....

💬 Answer: promoting healthy eating habits, improving nutrition and enhancing the well-being of children.


In [12]:
answer, docs = ask_rag("Who is the target audience of this report?")


🔎 Question: Who is the target audience of this report?

📎 Retrieved 3 chunks:

  --- Chunk 1 (Page 65) ---
  Health Organization; 2024 (https://gifna.who.int/).
105. Global nutrition policy review: what does it take to scale up nutrition action? Geneva: World Health 
Organization; 2013 (https://iris.who.int/...

  --- Chunk 2 (Page 5) ---
  Annex 5.  External contributors 54
Annex 6.  Guidance questions for the review of contextual factors 56
Annex 7.  Details of rapid review update 57
Annex 8.  GRADE evidence profiles 63
  GRADE evidenc...

  --- Chunk 3 (Page 32) ---
  3. Summary of evidence...

💬 Answer: I don't have enough information to answer this.


In [13]:
# Try your own question
my_question = "What health issues are discussed?"
answer, docs = ask_rag(my_question)


🔎 Question: What health issues are discussed?

📎 Retrieved 3 chunks:

  --- Chunk 1 (Page 21) ---
  behaviour; provision of health education about nutrition; screening for nutritional problems; referral and support for 
management of anaemia (e.g. iron supplementation); iron, folic acid and other mi...

  --- Chunk 2 (Page 25) ---
  behaviour; provision of health education about nutrition; screening for nutritional problems; referral and support for 
management of anaemia (e.g. iron supplementation); iron, folic acid and other mi...

  --- Chunk 3 (Page 39) ---
  understanding of policy details, financial issues (including lack of financial resources, a perceived higher 
cost of healthy food and (mostly unfounded) arguments about reduced revenue or profits), l...

💬 Answer: nutrition, screening for nutritional problems, referral and support for management of anaemia (e.g. iron supplementation); iron, folic acid and other micronutrient supplementation; referral and support for overweig